# M2 QLoRA training (Kaggle T4)
Thin runner: all logic lives in `src/training/train_qlora.py`. Settings: GPU T4 x2 (only one is used), Internet ON, Secrets `HF_TOKEN` (write) + `WANDB_API_KEY`.
Run cells in order. Do not start the full run until `--check` and the smoke run pass.

In [ ]:
# 1. Clone an exact commit (it must be pushed; if you change the script, push and paste the new SHA).
COMMIT = "6bf9ad6dd2012d700251e0a89ce92dee7b97f7f4"
!git clone https://github.com/hlong24082k/Diff-Mind
%cd Diff-Mind
!git checkout $COMMIT
!git log --oneline -1

In [ ]:
# 2. Install. Pins = versions the v1 run used (outputs/v1/versions.txt). torch 2.11.0+cu128 came preinstalled on Kaggle, so it is not pinned.
!pip install -q unsloth==2026.10.11 unsloth_zoo==2026.10.11 trl==1.15.0 peft==0.20.0 transformers==5.16.1 bitsandbytes==0.50.2 accelerate==1.14.0
!pip install -q pyyaml pydantic

In [ ]:
# 3. Secrets + single GPU. Must run BEFORE anything imports torch (restart the kernel if you already did).
import os
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
os.environ["HF_TOKEN"] = s.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = s.get_secret("WANDB_API_KEY")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# 4. Pre-flight: masking / EOS / length asserts, then exit. Read the decoded masked split yourself.
!python -m src.training.train_qlora --check

In [ ]:
# 5. Smoke run: 10 steps, checks VRAM + finite loss. Never pushes to the Hub (push_to_hub is off when --max-steps is set).
!python -m src.training.train_qlora --max-steps 10 --run-name v1-smoke

In [ ]:
# 6. Full run v1 (~200 steps). First drop the smoke run's leftovers: the push uploads all of outputs/v1.
!rm -rf outputs/v1/checkpoint-* outputs/v1/sanity.jsonl outputs/v1/train_summary.json
!python -m src.training.train_qlora

In [ ]:
# 6b. Verify the run: summary, sanity, and what actually landed on the Hub.
import json
from huggingface_hub import HfApi

print(open("outputs/v1/train_summary.json").read())
rows = [json.loads(l) for l in open("outputs/v1/sanity.jsonl")]
print("stopped", sum(r["stopped"] for r in rows), "valid", sum(r["valid"] for r in rows), "of", len(rows))
for f in HfApi().model_info("HLeon/diff-mind-qwen2.5-coder-3b-lora", files_metadata=True).siblings:
    print(f"{(f.size or 0) / 2**20:8.1f} MiB  {f.rfilename}")

In [ ]:
# 7. Record library versions, then paste them into cell 2 as pins (6.2).
!mkdir -p outputs/v1
!pip freeze | grep -iE "unsloth|trl|peft|transformers|bitsandbytes|torch|accelerate" > outputs/v1/versions.txt
!cat outputs/v1/versions.txt